In [ ]:
from typing import TypedDict
from langchain.chat_models import init_chat_model
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from dotenv import load_dotenv

load_dotenv()

# =================0.先初始化一个模型================
llm = init_chat_model(
    mode = 'deepseek-flash',
    extra_body = {'thinking': {'type': 'disabled'}}
)

# ===========1.定义State，记录用户输入和LLM返回的结果=========
class SimpleAgentState(TypedDict):
    user_input: str
    result: str

# ===============2.定义Node，实现LLM调用===============
def call_llm(state: SimpleAgentState):
    # 调用llm，得到结果
    response = llm.invoke(state['user_input'])
    # 更新state
    return {'result': response.text}

# =================3.创建Graph===============
builder = StateGraph(SimpleAgentState)
builder.add_node('llm', call_llm)
builder.add_edge(START, 'llm')
builder.add_edge('llm', END)
graph = builder.compile()

# ================4.调用Graph===================
result = graph.invoke({'user_input': '你好'})
print(result)